# 命名空间与旧项目阅读

学习目标：读懂命名空间、全局声明和三斜线依赖，识别 TypeScript 7 无法继续使用的旧项目配置。

前置知识：模块作用域、声明文件、接口合并、脚本与模块的区别。

适用版本与条件：TypeScript 7.0.2、Node.js 24.11.0；strict，模块例与全局脚本例分别配置，历史错误不进入正常构建。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/28-legacy-namespaces/。

1. [main.ts](scripts/28-legacy-namespaces/main.ts)：模块内部的命名空间、合并和别名。
2. [ambient.d.ts](scripts/28-legacy-namespaces/ambient.d.ts)：只包含类型的环境命名空间。
3. [global-a.ts](scripts/28-legacy-namespaces/global-a.ts)：供另一文件引用的全局接口。
4. [global-b.ts](scripts/28-legacy-namespaces/global-b.ts)：三斜线依赖和可运行的全局命名空间。
5. [tsconfig.global.json](scripts/28-legacy-namespaces/tsconfig.global.json)：独立的全局脚本编译范围。
6. [historical.ts](scripts/28-legacy-namespaces/historical.ts)：仅用于旧语法阅读和预期诊断。
7. [tsconfig.historical-output.json](scripts/28-legacy-namespaces/tsconfig.historical-output.json)：仅用于核对已移除选项的配置。
8. [tsconfig.json](scripts/28-legacy-namespaces/tsconfig.json)：模块例的正常配置；预期旧语法错误使用 tsconfig.errors.json。

## 1 命名空间组织同一作用域里的名字

namespace 把相关声明放进一个命名范围。成员加 export 才能从外部通过限定名访问；这里 Text 是 Toolkit 的嵌套命名空间。包含运行值的命名空间会生成 JavaScript 对象及初始化逻辑，不是单纯的类型擦除。

main.ts 末尾的 export {} 让整个文件保持模块作用域，因此 Toolkit 不成为其他模块自动可见的全局名字。命名空间与 ES 模块不是同一种组织机制；新代码通常通过模块导入导出来表达文件边界。

以下片段来自 main.ts。

```typescript
namespace Toolkit {
  export namespace Text {
    export function label(value: string): string { return "[" + value + "]"; }
  }
}
```

Step 1：检查本章正常项目。

```bash
npm run check:28
# 无类型诊断。
```

Step 2：生成当前源码的 JavaScript。

```bash
npm run build:28
# 类型错误时不生成新输出。
```

Step 3：执行本章运行入口。

```bash
npm run run:28
# 模块例输出 [ADA] 1；全局例输出 ADA。
```

## 2 合并声明和命名空间别名

同一作用域中的两个 Toolkit 声明合并，第二块增加 version；各块的未导出局部变量不因此全部互相可见。import Text = Toolkit.Text 是 TypeScript 的别名语法，可同时指向命名空间对应的类型与运行值，不是从文件加载模块。

下面是同一 main.ts 的后半部分。别名最终能在运行代码中访问 Text.label，是因为 Text 确实包含运行时函数。

```typescript
namespace Toolkit {
  export const version = "1";
}
import Text = Toolkit.Text;
const settings: LegacySettings.Options = { upper: true };
console.log(Text.label(settings.upper ? "ADA" : "Ada"), Toolkit.version); // [ADA] 1。
export {};
```

## 3 环境命名空间不创建运行对象

declare namespace 描述已经存在的命名空间或只用于类型的名字。本例只声明 Options 接口，所以 main.ts 的设置对象可以使用 LegacySettings.Options 检查；输出不会出现 LegacySettings 的构造或初始化。

如果声明里再加入函数或常量，并在代码里调用它们，就必须由实际宿主脚本或模块提供相同实现。环境声明本身不会加载脚本，也不能把不存在的运行值变出来。

以下片段来自 ambient.d.ts。

```typescript
declare namespace LegacySettings {
  interface Options { upper: boolean }
}
```

## 4 跨文件声明与三斜线指令

全局脚本可以在多文件中合并同名命名空间，前提是这些文件都进入同一个检查项目且没有被当成独立模块。本例用 moduleDetection: legacy 明确阅读旧式脚本；配置只列 global-b.ts，path 指令把 global-a.ts 加入编译。

三斜线指令必须位于文件顶部，只能有注释或其他指令在前面。path 的相对路径相对于包含它的文件，预处理按根文件顺序和引用出现顺序深度优先展开；这影响编译输入，不等于宿主加载顺序。

以下片段来自 global-a.ts。

```typescript
namespace GlobalLesson {
  export interface Item { name: string }
}
```

以下片段来自 global-b.ts。

```typescript
/// <reference path="./global-a.ts" />
/// <reference types="node" />
/// <reference lib="es2025" />
namespace GlobalLesson {
  export function label(item: Item): string { return item.name.toUpperCase(); }
}
const globalItem: GlobalLesson.Item = { name: "Ada" };
console.log(GlobalLesson.label(globalItem)); // ADA；global-a.ts 只有被擦除的接口。
```

| 指令名称 | 中文名称／含义 | 本例用途 |
| --- | --- | --- |
| reference path | 源文件依赖 | 加入 global-a.ts |
| reference types | 类型包依赖 | 载入 node 声明以识别 console |
| reference lib | 内置声明库依赖 | 指定 es2025 声明 |

types 不是运行时 import，lib 也不提供 polyfill。这里 global-a.ts 仅有接口，不产生需要先加载的运行值，所以执行 global-b.js 就足够。如果把运行函数分散到多个全局脚本，必须另行安排宿主加载；Node 模块包装不会因为编译器合并名字就共享局部变量。

以下片段来自 tsconfig.global.json。

```json
{
  "compilerOptions": {
    "target": "ES2025",
    "module": "ESNext",
    "moduleResolution": "bundler",
    "moduleDetection": "legacy",
    "strict": true,
    "types": [],
    "lib": [
      "ES2025"
    ],
    "rootDir": ".",
    "outDir": ".global",
    "noEmitOnError": true
  },
  "files": [
    "global-b.ts"
  ]
}
```

## 5 识别历史语法与已移除输出

旧书中的 module Historical 大括号形式曾表示内部模块，后来改称 namespace。它不同于声明文件中的 declare module "包名"：后者是外部模块声明，不是这个历史命名空间关键字用法。TypeScript 7 已禁止前一种写法，以下文件仅用于阅读与预期诊断。

TS 7 也不再支持 module 的 AMD、UMD、System 和 none 输出，以及 classic、node10 解析和 ES5 target 等已移除选项。历史三斜线配合 outFile 拼接输出的例子不能不加判断地复制到本章 NodeNext 项目。

先识别旧构建器和宿主需要的格式，再迁移到 ESM/CJS 或由合适构建器生成目标格式，补齐运行入口与加载关系；不要仅靠忽略弃用诊断维持旧配置。

以下片段来自 historical.ts。

```typescript
// 历史阅读：module 的这种命名空间用法已在 TypeScript 7 移除。
module Historical {
  export const value = 1;
}
```

以下片段来自 tsconfig.historical-output.json。

```json
{
  "extends": "./tsconfig.json",
  "compilerOptions": {
    "module": "AMD",
    "moduleResolution": "classic",
    "noEmit": true
  },
  "files": [
    "main.ts",
    "ambient.d.ts"
  ]
}
```

Step 1：核对旧 module 命名空间语法。

```bash
npm run errors:28
# 退出 1；TS1540 要求使用 namespace。
```

Step 2：核对旧输出及解析选项。

```bash
npm run errors:28:output
# 退出 1；TS5108 指出 AMD、Classic 已移除，另有 TS5095 配置组合诊断。
```

## 本章小结

- 命名空间可包含类型和运行值，declare 只提供声明。
- 三斜线表达编译依赖，不能代替宿主加载脚本。
- 历史 module 命名空间和旧输出配置在 TS 7 有明确限制，迁移需同时核对输出与宿主。

## 练习

1. 在 Toolkit 中增加导出的字符串函数，通过 Text 别名之外的限定名调用并核对结果。
2. 将 global-a.ts 的 Item 接口增加必需字段 id，确认 global-b.ts 因缺字段产生错误；补齐后两个项目都通过。
3. 把 historical.ts 的 module 改为 namespace，确认 TS1540 消失；说明这个修改为什么不会同时修复 AMD 输出配置。

### 提示

1. 成员放在 Toolkit 内并加 export，使用 Toolkit.成员名。
2. Item 来自全局脚本项目，先检查该项目的对象字面量。
3. 源码关键字和 compilerOptions 分别决定不同检查。

### 参考解析

1. 例如导出 `upper(value: string)` 并返回 `value.toUpperCase()`，调用 `Toolkit.upper("Ada")` 得到 ADA；未导出成员不能从外部限定访问。
2. Item 增加 `id: number` 后，globalItem 必须提供如 `id: 1`；模块项目中的 LegacySettings 不受该接口修改影响。补齐后正常项目均可通过。
3. namespace 修复历史语法诊断；AMD 与 classic 属于另一份配置的已移除选项，仍需分别迁移，不能靠改源码关键字恢复旧输出支持。

## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [Namespaces](https://www.typescriptlang.org/docs/handbook/namespaces.html) 的导出、别名、跨文件与环境命名空间；[Triple-Slash Directives](https://www.typescriptlang.org/docs/handbook/triple-slash-directives.html) 的位置、path/types/lib 和预处理顺序。手册中的旧输出例需结合当前版本限制阅读。 |
| Microsoft Developer Blogs | [TypeScript 7.0 / Updates Since 5.x, and New Behaviors from 6.0](https://devblogs.microsoft.com/typescript/announcing-typescript-7-0/#updates-since-5-x-and-new-behaviors-from-6-0)：已移除语法、输出和解析选项。 |
